# Deep learning for molecular SMILES: train and deploy on DataRobot

- **Author**: senkin.zhan@datarobot.com
- **Sample data**: [`input/train.csv`](input/train.csv) (training) · [`input/test.csv`](input/test.csv) (held-out)

## Summary

SMILES (Simplified molecular input line entry system) is a textual representation of molecular
structures. The companion accelerator *Feature engineering for molecular SMILES* turns those
strings into tabular features and hands them to DataRobot Autopilot. This accelerator takes the
other route: it learns the representation end-to-end with a PyTorch model that reads the molecule
directly — either as a **molecular graph** (DMPNN over RDKit-built graphs) or as a **character
sequence** (LSTM / CNN / Transformer over a SMILES tokenizer).

The point is that **both halves run on DataRobot**. A custom deep learning architecture normally
lives outside the platform — trained somewhere else, served by a hand-rolled stack. Here the whole
lifecycle stays in one place:

- **Train on DataRobot** — this notebook runs as-is in a DataRobot codespace or on DataRobot
  Notebooks, on CPU or on a GPU environment.
- **Deploy on DataRobot** — `deploy.ipynb` packages the trained PyTorch model as a **custom
  inference model**, registers it in the Model Registry, and deploys it, so it is governed,
  monitored, and scored through the standard prediction API like any DataRobot model.

Everything is driven by a single config file, `config/config.yaml`. Set `model.type` to `dmpnn`,
`lstm`, `cnn`, `transformer`, or a `+`-joined combination such as `lstm+cnn+transformer`, and
rerun — no code changes. A GPU speeds training up but is not required.

This notebook outlines how to:

1. Load `input/train.csv` and carve a 4:1 holdout out of it (`input/test.csv` is never touched).
2. Featurize SMILES with RDKit — into PyTorch Geometric molecular graphs, or into token
   sequences with a character-level SMILES tokenizer.
3. **Stage 1** — train on the 80 % sub-train with early stopping on holdout MAE to find the
   epoch at which validation error bottoms out.
4. **Stage 2** — refit from scratch on the full training set for exactly that many epochs,
   replaying stage 1's learning-rate trajectory.
5. Save a self-contained bundle artifact (weights + config + atom map / tokenizer vocabulary) to
   `model/smiles_model.pth`, ready for `deploy.ipynb`.


## Setup

### Install and import libraries

In [1]:
# torch, rdkit, torch-geometric and PyYAML come from deploy/requirements.txt.
# torch-scatter is intentionally absent - DataRobot's dependency manager rejects the
# --find-links line its wheels need, and src/models/_scatter.py falls back to
# torch_geometric.utils.scatter, so the C++ extension is never required.
!pip install -q -r deploy/requirements.txt

In [2]:
# This notebook uses the plain-text tqdm, but some third-party imports below
# pull in tqdm.auto themselves, which probes for ipywidgets and warns
# (TqdmWarning: IProgress not found) on kernels without it, e.g. DataRobot
# Notebooks. Filter that category before anything else is imported.
import warnings

from tqdm import tqdm, TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

import numpy as np
import pandas as pd
from src.config import load_config
from src.utils import seed_everything

cfg = load_config("config/config.yaml")
seed_everything(cfg["seed"])

target = cfg["target"]
model_type = cfg["model"]["type"]
smiles_col = cfg["data"]["smiles_column"]
print(f"Target: {target}, Model: {model_type}")

Random seed set to 42 for reproducible results
Target: Tc, Model: dmpnn


### Import data

`train.csv` holds the training rows; `split_train_valid` carves the stage-1 holdout out of it at
`data.valid_ratio` (0.2 → 4:1), seeded by `seed` so the split is reproducible.

In [3]:
%%time
from src.data.loader import load_split, split_train_valid

# train.csv = old train + old valid, merged. test.csv is left alone.
train_df, test_df = load_split(cfg)

# Stage-1 holdout: 4:1, seeded by cfg['seed'] so this is reproducible.
sub_train_df, valid_df = split_train_valid(train_df, cfg)

Loaded data from input/:
  train: 736, test: 131
Split train.csv 80%/20%: train 589, valid 147
CPU times: user 116 ms, sys: 28.5 ms, total: 144 ms
Wall time: 591 ms


## Featurize SMILES

The atom map / tokenizer is fit once on `train.csv` only, so held-out `test.csv`
never contributes atom types or characters. Unseen symbols at score time land
on a zero one-hot / `<UNK>`. Featurizing the full `train_df`
once and then indexing into it keeps stage 1 and stage 2 bit-identical on shared rows.

In [4]:
%%time
from src.training.trainer import _parse_model_type

is_graph, _ = _parse_model_type(model_type)
tokenizer = None
atom_map = None
max_length = cfg["data"]["max_length"]

if is_graph:
    from src.data.graph_features import smiles_to_graph
    from src.data.loader import build_atom_map

    atom_map = build_atom_map(cfg)

    def make_data(df):
        return [
            smiles_to_graph(row[smiles_col], atom_map, row[target])
            for _, row in tqdm(df.iterrows(), total=len(df))
        ]

    print("Converting to graphs...")
else:
    from src.data.loader import build_tokenizer
    from src.data.smiles_data import SMILESDataset

    tokenizer = build_tokenizer(cfg)

    def make_data(df):
        return SMILESDataset(
            df[smiles_col].tolist(), df[target].tolist(), tokenizer, max_length=max_length
        )


stage1_train_data = make_data(sub_train_df)
stage1_valid_data = make_data(valid_df)
full_train_data = make_data(train_df)

# dmpnn infers its input dim from a sample Data object; sequence models ignore this.
stage1_data_list = stage1_train_data if is_graph else None
full_data_list = full_train_data if is_graph else None

print(f"stage 1 -> train: {len(stage1_train_data)}, valid: {len(stage1_valid_data)}")
print(f"stage 2 -> train: {len(full_train_data)}")

Converting to graphs...
100%|██████████| 736/736 [00:03<00:00, 227.33it/s]stage 1 -> train: 589, valid: 147
stage 2 -> train: 736
CPU times: user 7.27 s, sys: 138 ms, total: 7.4 s
Wall time: 7.02 s



## Stage 1 — Find the best epoch count

Early stopping on holdout MAE. The checkpoint goes to `model/smiles_model_stage1.pth` so it
does not clobber the deployable artifact that stage 2 writes.

In [5]:
%%time
from src.training.trainer import run_train

stage1 = run_train(
    stage1_train_data,
    stage1_valid_data,
    cfg,
    data_list=stage1_data_list,
    tokenizer=tokenizer,
    atom_map=atom_map,
    artifact_name=cfg["paths"].get("stage1_artifact_name", "smiles_model_stage1.pth"),
)

best_epoch = stage1["best_epoch"]
print(f"\nStage 1: best epoch = {best_epoch}, holdout MAE = {stage1['best_val_mae']:.4f}")

train: 589, valid: 147
Epoch: 01, Train Loss: 0.9140, Valid MAE: 0.0792, LR: 1.00e-03
  -> New best: 0.0792
Epoch: 02, Train Loss: 0.0908, Valid MAE: 0.0573, LR: 1.00e-03
  -> New best: 0.0573
Epoch: 03, Train Loss: 0.0760, Valid MAE: 0.0607, LR: 1.00e-03
Epoch: 04, Train Loss: 0.0703, Valid MAE: 0.0423, LR: 1.00e-03
  -> New best: 0.0423
Epoch: 05, Train Loss: 0.0649, Valid MAE: 0.0497, LR: 1.00e-03
Epoch: 06, Train Loss: 0.0531, Valid MAE: 0.0374, LR: 1.00e-03
  -> New best: 0.0374
Epoch: 07, Train Loss: 0.0570, Valid MAE: 0.0360, LR: 1.00e-03
  -> New best: 0.0360
Epoch: 08, Train Loss: 0.0519, Valid MAE: 0.0479, LR: 1.00e-03
Epoch: 09, Train Loss: 0.0526, Valid MAE: 0.0391, LR: 1.00e-03
Epoch: 10, Train Loss: 0.0526, Valid MAE: 0.0391, LR: 1.00e-03
Epoch: 11, Train Loss: 0.0572, Valid MAE: 0.0516, LR: 1.00e-03
Epoch: 12, Train Loss: 0.0472, Valid MAE: 0.0359, LR: 1.00e-03
  -> New best: 0.0359
Epoch: 13, Train Loss: 0.0431, Valid MAE: 0.0387, LR: 1.00e-03
Epoch: 14, Train Loss: 0.0

In [6]:
valid_df["preds"] = stage1["valid_preds"]
valid_df[[target, "preds"]].describe()

,Tc,preds
count,147.000000,147.000000
mean,0.258331,0.254028
std,0.086376,0.075212
min,0.069000,0.099546
25%,0.197000,0.194524
50%,0.243000,0.251440
75%,0.320083,0.321407
max,0.460000,0.393351


## Stage 2 — Refit on the full training set

Retrains from scratch on every row for exactly `best_epoch` epochs — no validation set, so no
early stopping and no "best checkpoint" to pick; the weights at the final epoch are what gets
saved. `lr_schedule` replays stage 1's per-epoch learning rate so the run that produced
`best_epoch` is reproduced as closely as possible.

Note the epoch count is used **as-is**. The full set is ~25 % larger than stage 1's sub-train,
so each epoch is ~25 % more gradient steps.

In [7]:
%%time
from src.training.trainer import run_train_full

seed_everything(cfg["seed"])

result = run_train_full(
    full_train_data,
    cfg,
    epochs=best_epoch,
    data_list=full_data_list,
    tokenizer=tokenizer,
    atom_map=atom_map,
    lr_schedule=stage1["lr_history"],
    artifact_name=cfg["paths"].get("artifact_name", "smiles_model.pth"),
)

print(f'\nModel saved to: {result["best_model_path"]}')

Random seed set to 42 for reproducible results
Refit on full train set: 736 samples, 93 epochs
Replaying stage-1 LR schedule (1.00e-03 -> 5.00e-04)
Epoch: 01/93, Train Loss: 0.6901, LR: 1.00e-03
Epoch: 02/93, Train Loss: 0.0721, LR: 1.00e-03
Epoch: 03/93, Train Loss: 0.0746, LR: 1.00e-03
Epoch: 04/93, Train Loss: 0.0617, LR: 1.00e-03
Epoch: 05/93, Train Loss: 0.0590, LR: 1.00e-03
Epoch: 06/93, Train Loss: 0.0562, LR: 1.00e-03
Epoch: 07/93, Train Loss: 0.0483, LR: 1.00e-03
Epoch: 08/93, Train Loss: 0.0484, LR: 1.00e-03
Epoch: 09/93, Train Loss: 0.0487, LR: 1.00e-03
Epoch: 10/93, Train Loss: 0.0498, LR: 1.00e-03
Epoch: 11/93, Train Loss: 0.0509, LR: 1.00e-03
Epoch: 12/93, Train Loss: 0.0461, LR: 1.00e-03
Epoch: 13/93, Train Loss: 0.0430, LR: 1.00e-03
Epoch: 14/93, Train Loss: 0.0434, LR: 1.00e-03
Epoch: 15/93, Train Loss: 0.0452, LR: 1.00e-03
Epoch: 16/93, Train Loss: 0.0409, LR: 1.00e-03
Epoch: 17/93, Train Loss: 0.0376, LR: 1.00e-03
Epoch: 18/93, Train Loss: 0.0396, LR: 1.00e-03
Epoch:

In [8]:
train_df["preds"] = result["train_preds"]
train_df[[target, "preds"]].describe()

,Tc,preds
count,736.000000,736.000000
mean,0.256823,0.258708
std,0.103362,0.091308
min,0.046500,0.057939
25%,0.186375,0.188639
50%,0.236000,0.243561
75%,0.325250,0.335021
max,1.590000,0.696902


## Next steps

`model/smiles_model.pth` now holds the trained bundle.

1. Run `deploy.ipynb` to flatten `src/` into `deploy/model_lib.py`, upload it with the DRUM hooks
   as a custom inference model, build dependencies, register, and deploy.
2. Run `predict.ipynb` to score `input/test.csv` through the deployment and benchmark the result.
3. To try another architecture, edit `model.type` in `config/config.yaml` and rerun this notebook.